**Case 4: Caroline Bezengi oil spill, Hallaniyat Islands, Oman (June to August 2026)** <br/>

On 8 June 2026, an explosion disabled the Caroline Bezengi, a sanctioned tanker carrying close to a million barrels of Russian crude to India. The crew was taken off about four days later, and around 23 June the tanker ended up on the rocks at the southwest tip of Jazirat Al Qibliyyah, one of the Hallaniyat Islands off the Dhofar coast of Oman ([Maritime Executive](https://maritime-executive.com/article/stricken-tanker-caroline-bezengi-continues-to-leak-oil-off-oman))

It leaked through the Khareef monsoon, when strong winds and waves kept salvage crews away. On 10 August Oman put the slick at about 390 km² ([Reuters via gCaptain](https://gcaptain.com/oil-spill-off-oman-from-tanker-under-sanctions-against-russia-spreads-over-huge-area/)), and by 13 August oil had reached the beaches at Ras Madrakah, up to 40 km of coastline ([Al Jazeera](https://www.aljazeera.com/news/2026/8/13/oman-says-oil-spill-from-stricken-tanker-has-reached-its-coast))

This notebook looks at what NISAR saw: when the tanker shows up on the rocks, how the slick grows from one pass to the next, and where it drifts

**Main sources** <br/>

- [Challenge resources](https://www.spaceappschallenge.org/2026/challenges/dancing-with-the-sars/?tab=resources)

- [NASA Earthdata Search](https://search.earthdata.nasa.gov/search)

- [CMR search API](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html)

- [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/)

- [Earthdata Login](https://urs.earthdata.nasa.gov/)

**Summary** <br/>

- 
- 
- 
- 


<br/>
<br/>

### 1. Data Retrieval (API Connection)

Find the NISAR data for this case through **[CMR (Common Metadata Repository)](https://www.earthdata.nasa.gov/about/esdis/eosdis/cmr)**, NASA's catalogue of every Earthdata dataset ([Earthdata Search](https://www.earthdata.nasa.gov/learn/earthdata-search) runs on it), then check if we can read the actual files

- Searching CMR is public, no login needed ([CMR search API](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html))

- The data files need an Earthdata login ([Accessing NISAR data](https://nisar-docs.asf.alaska.edu/access-overview))

#### 1.1 NISAR collections

- **Level 1 (in radar coordinates)**: radar geometry, not on a map yet

- **Level 2 (in geographic coordinates)**: put on a map grid and terrain corrected, with a quality mask

- **Level 3 (derived geophysical parameters)**: a physical quantity worked out from Level 2, e.g. soil moisture

([NISAR data products](https://nisar-docs.asf.alaska.edu/products-overview))

<br/>

Each dataset in CMR is a **collection** with a short name. A keyword search for "NISAR" lists them all ([keyword search](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html#c-keyword)), then keep the Level 2 and Level 3 ones

In [1]:
# view NISAR collections
import requests
import pandas as pd
pd.set_option("display.max_colwidth", 80)

CMR = "https://cmr.earthdata.nasa.gov/search"

resp = requests.get(f"{CMR}/collections.json", params={"keyword": "NISAR", "page_size": 100}, timeout=60)
entries = resp.json()["feed"]["entry"]

collections = pd.DataFrame([(e["short_name"], e["title"], e["time_start"][:10]) for e in entries],
                           columns=["short_name", "title", "data_starts"])

# keep the level 2 and 3 science products, plus urgent response
collections = collections[collections.short_name.str.match(r"NISAR_(L2|L3|UR_L2)")].sort_values("short_name")
collections

,short_name,title,data_starts
8,NISAR_L2_GCOV_BETA_V1,NISAR Beta Geocoded Polarimetric Covariance Product (Version 1),2025-10-01
0,NISAR_L2_GCOV_PROVISIONAL_V1,NISAR Provisional Geocoded Polarimetric Covariance Product (Version 1),2025-10-28
22,NISAR_L2_GCOV_V1,NISAR Validated Geocoded Polarimetric Covariance Product (Version 1),2025-10-28
14,NISAR_L2_GOFF_BETA_V1,NISAR Beta Geocoded Pixel Offsets (Version 1),2025-10-01
5,NISAR_L2_GOFF_PROVISIONAL_V1,NISAR Provisional Geocoded Pixel Offsets (Version 1),2025-10-28
23,NISAR_L2_GOFF_V1,NISAR Validated Geocoded Pixel Offsets (Version 1),2025-10-28
11,NISAR_L2_GSLC_BETA_V1,NISAR Beta Geocoded Single Look Complex Product (Version 1),2025-10-01
1,NISAR_L2_GSLC_PROVISIONAL_V1,NISAR Provisional Geocoded Single Look Complex Product (Version 1),2025-10-28
24,NISAR_L2_GSLC_V1,NISAR Validated Geocoded Single Look Complex Product (Version 1),2025-10-28
12,NISAR_L2_GUNW_BETA_V1,NISAR Beta Geocoded Unwrapped Interferogram Product (Version 1),2025-10-01


- Each product comes in up to three quality stages: **Beta** (early, not for science), **Provisional** (calibrated, partly validated, from July 20, 2026) and **Validated** (fully calibrated and validated, released November 6, 2026) ([available data](https://nisar-docs.asf.alaska.edu/availability-overview))

- The spill runs from June to August 2026, so both **Provisional** and **Validated** are worth checking, compared in 1.2

<br/>

The products most likely to be useful for this case:

- **GCOV**: radar backscatter for each pass. Oil calms the small ripples on the sea surface, so less of the radar pulse bounces back and the slick shows up dark, while a ship shows up as a bright spot ([SEOS, oil on radar](https://seos-project.eu/marinepollution/marinepollution-c02-s02-p03.html))

- **GUNW**: coherence between two passes 12 days apart. Open water changes constantly, so it has almost no coherence, this is checked rather than assumed

- **Urgent Response**: passes processed fast after a disaster

([NISAR data products](https://nisar-docs.asf.alaska.edu/products-overview))

<br/>

#### 1.2 Granules over the study area

A **granule** is one file: one pass over one area for GCOV, or one pair of passes for GUNW

The search below asks CMR for every granule of a collection that touches our box within a date range

- `bounding_box`: `west,south,east,north` in degrees

- `temporal`: start and end date

- `.umm_json`: returns the full record for each granule, including the download links

([granule search](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html#granule-search-by-parameters))

<br/>

Study area: from the wreck at Jazirat Al Qibliyyah and the rest of the Hallaniyat Islands, north-east along the coast to Ras Madrakah, where the oil came ashore, drawn on [bboxfinder](http://bboxfinder.com/#17.20,55.80,19.20,58.00). Window: May to October 2026, from before the explosion on 8 June to well after the oil reached the coast

Place positions are from OpenStreetMap ([OpenStreetMap](https://www.openstreetmap.org/)) and Wikipedia for the islands ([Hallaniyat Islands](https://en.wikipedia.org/wiki/Khuriya_Muriya_Islands))

In [2]:
import plotly.graph_objects as go

AOI = (55.80, 17.20, 58.00, 19.20)   # Area of Interest: west, south, east, north
west, south, east, north = AOI

# places to label on the map
places = pd.DataFrame([
    ("Jazirat Al Qibliyyah (wreck)", 56.3375, 17.5000),
    ("Al Hallaniyah", 56.0153, 17.5168),
    ("Shuwaymiyah", 55.6097, 17.8853),
    ("Sharbithat", 56.2721, 17.9385),
    ("Ras Madrakah", 57.7824, 18.9806),
], columns=["name", "lon", "lat"])

# satellite basemap with the study area and places drawn on top
fig = go.Figure(go.Scattermap(lon=[west, east, east, west, west], lat=[south, south, north, north, south],
                              mode="lines", line=dict(color="red", width=2), name="study area"))
fig.add_trace(go.Scattermap(lon=places.lon, lat=places.lat, mode="markers+text", text=places.name, textposition="top right",
                            marker=dict(size=9, color="yellow"), textfont=dict(color="white"), name="places"))
fig.update_layout(
    map=dict(style="white-bg", center=dict(lon=(west + east) / 2, lat=(south + north) / 2), zoom=7,
             layers=[dict(below="traces", sourcetype="raster", sourceattribution="Esri World Imagery",
                          source=["https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"])]),
    height=500, margin=dict(l=0, r=0, t=0, b=0), showlegend=False)
fig.show()

In [3]:
START, END = "2026-05-01", "2026-10-31"

# function to find every NISAR file over the study area within the date window
def search_granules(short_name, bbox=AOI, start=START, end=END):
    """
    Search CMR for every granule of one collection over a box and date range

    Args:
        short_name: Collection short name, e.g. "NISAR_L2_GCOV_PROVISIONAL_V1"
        bbox: Bounding box (west, south, east, north) in degrees
        start: First date to include, "YYYY-MM-DD"
        end: Last date to include, "YYYY-MM-DD"

    Returns:
        DataFrame with one row per granule: name, start and end time, size in GB, and the .h5 download link
    """
    rows, page = [], 1
    while True:
        params = {"short_name": short_name, "bounding_box": ",".join(map(str, bbox)),
                  "temporal": f"{start}T00:00:00Z,{end}T23:59:59Z", "page_size": 200, "page_num": page}
        items = requests.get(f"{CMR}/granules.umm_json", params=params, timeout=120).json()["items"]
        for item in items:
            umm = item["umm"]
            urls = [u["URL"] for u in umm["RelatedUrls"] if u["URL"].startswith("https")]
            rows.append({
                "granule": umm["GranuleUR"],
                "start": pd.to_datetime(umm["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]),
                "end": pd.to_datetime(umm["TemporalExtent"]["RangeDateTime"]["EndingDateTime"]),
                "size_gb": round(umm["DataGranule"]["ArchiveAndDistributionInformation"][0]["SizeInBytes"] / 1e9, 2),
                "url": next(u for u in urls if u.endswith(".h5")),
            })
        # CMR returns at most 200 per page, so a short page means we have them all
        if len(items) < 200:
            break
        page += 1
    return pd.DataFrame(rows, columns=["granule", "start", "end", "size_gb", "url"])

In [4]:
# search each collection over the study area for May to October
gcov = search_granules("NISAR_L2_GCOV_PROVISIONAL_V1")
gcov_validated = search_granules("NISAR_L2_GCOV_V1")
gunw = search_granules("NISAR_L2_GUNW_PROVISIONAL_V1")
urgent = search_granules("NISAR_UR_L2")

# how many files each one has, and how much it would be to download them all
found = {"GCOV Provisional": gcov, "GCOV Validated": gcov_validated, "GUNW Provisional": gunw, "Urgent Response": urgent}
pd.DataFrame({"granules": [len(d) for d in found.values()],
              "total_gb": [round(d.size_gb.sum(), 1) for d in found.values()],
              "first": [d.start.min() for d in found.values()],
              "last": [d.start.max() for d in found.values()]}, index=list(found))

,granules,total_gb,first,last
GCOV Provisional,91,103.3,2026-06-17 14:40:56+00:00,2026-10-21 01:33:24+00:00
GCOV Validated,6,10.7,2026-10-22 14:49:12+00:00,2026-10-28 01:41:24+00:00
GUNW Provisional,57,99.9,2026-06-17 14:40:56+00:00,2026-10-09 01:33:24+00:00
Urgent Response,0,0.0,NaT,NaT


- **GCOV Provisional**: 91 passes from 17 June to 21 October, about 103 GB. Five different stacks cross the box, so there's a pass almost every one to six days, apart from the instrument gap from Jul 27 to Aug 10 ([available data](https://nisar-docs.asf.alaska.edu/availability-overview))

- **GCOV Validated**: only from 22 October over this box, after the spill, so **Provisional** is the one to use

- **GUNW**: 57 pairs, about 100 GB

- **Urgent Response**: none over the box

**Urgent Response**

No Urgent Response granules over the box, so check a much wider area in case the processing was set up around the coast instead

In [7]:
# urgent response over a much wider box, the whole Arabian Sea coast of Oman
urgent_wide = search_granules("NISAR_UR_L2", bbox=(52.0, 15.0, 60.0, 23.0))
print(f"urgent response granules over the wider box: {len(urgent_wide)}")
urgent_wide[["start", "granule"]]

urgent response granules over the wider box: 0


,start,granule


- None anywhere along the coast either. Unlike case 1, NISAR never switched this area to emergency processing, the spill was only picked up by the regular passes

- That also makes sense for an oil spill: NISAR mainly images land and sea ice ([About NISAR](https://nisar-docs.asf.alaska.edu/nisar-intro)), the sea around the islands is only covered because it sits next to the coast

<br/>

#### 1.3 Accessing the files

**Earthdata login**

The `.h5` files need an Earthdata Login token, saved in `dataScience/.env` as `EDL_TOKEN=...` ([how to get a token](https://urs.earthdata.nasa.gov/documentation/for_users/user_token))

In [8]:
from pathlib import Path

# read EDL_TOKEN from dataScience/.env
env = dict(line.split("=", 1) for line in Path("../.env").read_text().splitlines() if "=" in line and not line.startswith("#"))
AUTH = {"Authorization": f"Bearer {env['EDL_TOKEN'].strip()}"}

Check the token works by asking for just the first 8 bytes of one real file. Every HDF5 file starts with the same signature, `\x89HDF\r\n\x1a\n`

- `206` + that signature: the token works

- `200` with a web page: the request got sent to the login page, so the token is missing or expired

In [9]:
# check if the first GCOV file is accessible with the Earthdata login token
url = gcov.url.iloc[0]
check = requests.get(url, headers={**AUTH, "Range": "bytes=0-7"}, timeout=60)
check.status_code, check.content[:8]

(206, b'\x89HDF\r\n\x1a\n')

<br/>

**Retrieving a file**

The guide mentions files can be streamed instead of downloaded ([Accessing NISAR data](https://nisar-docs.asf.alaska.edu/access-overview)). The download link redirects to a temporary signed link to the file in the cloud, and `h5py` can open it as a file-like object through `fsspec` ([h5py](https://docs.h5py.org/en/stable/high/file.html#python-file-like-objects)), so only the parts we actually read come over the network

In [10]:
import h5py
import fsspec

# function to open a NISAR file in the cloud without downloading the whole thing
def open_h5(url):
    """
    Open a NISAR .h5 file remotely, reading only the parts that get used

    Args:
        url: The .h5 download link from search_granules

    Returns:
        h5py File that can be read like a local file
    """
    # the download link redirects to a temporary signed link, grab it instead of following it
    signed = requests.get(url, headers=AUTH, allow_redirects=False, timeout=60).headers["Location"]

    # fsspec fetches the file in 4 MB pieces, only when h5py asks for them
    return h5py.File(fsspec.open(signed, "rb", block_size=4 * 2**20).open(), "r")

In [11]:
# quick test: list the top-level groups of one GCOV file
with open_h5(gcov.url.iloc[0]) as f:
    print(list(f.keys()))
    print(list(f["science/LSAR"].keys()))

['science']
['GCOV', 'identification']


<br/>

**What each file holds**

Each file can hold two **frequencies**: frequency A, the main one, and frequency B, a narrower band mainly collected for atmospheric corrections. Frequency B has bigger pixels, so its files are smaller and quicker to read, which suits a quick look over a large area ([About NISAR](https://nisar-docs.asf.alaska.edu/nisar-intro), [GCOV](https://nisar-docs.asf.alaska.edu/gcov))

In [12]:
# for one pass of each frame and mode: which frequencies it holds, their polarizations and pixel size
frames = gcov.assign(frame=gcov.granule.str.extract(r"GCOV_\d{3}_(\d{3}_[AD]_\d{3}_\d{4}_[A-Z]{4})")[0]).drop_duplicates("frame")
rows = []
for r in frames.itertuples():
    with open_h5(r.url) as f:
        for freq, grid in f["science/LSAR/GCOV/grids"].items():
            x = grid["xCoordinates"]
            rows.append((r.frame, freq, [p.decode() for p in grid["listOfPolarizations"][()]],
                         float(x[1] - x[0]), x.shape[0], int(grid["projection"][()])))
pd.DataFrame(rows, columns=["frame", "frequency", "polarizations", "pixel_m", "width_px", "epsg"])

,frame,frequency,polarizations,pixel_m,width_px,epsg
0,049_D_079_2005_SHSH,frequencyA,[HH],20.0,17028,32640
1,049_D_079_2005_SHSH,frequencyB,[HH],80.0,4257,32640
2,056_A_010_0005_NASV,frequencyB,[VV],80.0,4266,32640
3,056_A_011_2005_DHDH,frequencyA,"[HH, HV]",20.0,17100,32640
4,056_A_011_2005_DHDH,frequencyB,"[HH, HV]",80.0,4275,32640
5,128_A_010_0005_NASV,frequencyB,[VV],80.0,4293,32640
6,128_A_011_0005_NASV,frequencyB,[VV],80.0,4302,32640
7,128_A_011_2005_DHDH,frequencyA,"[HH, HV]",20.0,17208,32640
8,128_A_011_2005_DHDH,frequencyB,"[HH, HV]",80.0,4302,32640
9,150_D_079_2005_DHDH,frequencyA,"[HH, HV]",20.0,17100,32640


- **Frequency A**: 20 m pixels in most frames (10 m for `157_A_010`), with HH and HV, or HH only for the `SHSH` passes

- **Frequency B**: 80 m pixels in **every** frame, HH (and HV) for most, and VV only for the `0005 NASV` frames, which have no frequency A at all

- Every frame is on the same map projection, **EPSG 32640** (WGS 84 / UTM zone 40N, [epsg.io](https://epsg.io/32640))

- The slick covers hundreds of km², so the 80 m frequency B layer is the one every pass has in common, and much quicker to read over the whole box. The 20 m layer is for a closer look at the wreck

We can now find every NISAR file over the spill and read them remotely. Next is section 2: lining up the passes from the different stacks and checking how consistent they are

<br/>
<br/>